# End-to-end IMC25 SOTA pipeline (run + metrics + visualizations)

This notebook is a **comprehensive end-to-end runner** for the pipeline in this repo.

This notebook is configured to **avoid using `data/test/`**.

**Evaluation protocol used here**:
- Treat these three datasets as the *test/holdout* split (but sourced from `data/train/<dataset>/`):
  - `ETs`
  - `pt_sacrecoeur_trevi_tajmahal`
  - `pt_stpeters_stpauls`
- Train retrieval on the remaining `data/train/*` datasets (excluding the holdout datasets).

It computes **official-style metrics** on the holdout split:
  - **pose-aware mAA** (camera centers with similarity alignment)
  - **clustering score** (precision-like)
  - **final score** (harmonic mean)

Visualizations included:
- cluster size distributions + outliers
- 2D embedding plot (PCA) colored by predicted clusters
- retrieval graph visualization (subsampled)
- 3D camera center visualization (colored by cluster)
- point cloud preview (requires `open3d`)

## Runtime notes
- `--preset sota` enables robust matching fallback (`colmap -> dino -> diffusion`) + dense MVS + depth fallback + demo export.
- First run may download models.
- Dense MVS + diffusion are compute-heavy.


In [1]:
from __future__ import annotations

import json
import os
import shlex
import struct
import subprocess
import sys
from dataclasses import dataclass
from pathlib import Path, PurePosixPath
from typing import Dict, Optional, Tuple

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)


def find_repo_root(start: Path | None = None) -> Path:
    start = start or Path.cwd()
    for p in [start] + list(start.parents):
        if (p / "pyproject.toml").exists() or (p / "PIPELINE.md").exists():
            return p
    return start


REPO_ROOT = find_repo_root()
print("REPO_ROOT:", REPO_ROOT)

# Make local src importable for helper utilities.
SRC = REPO_ROOT / "src"
if SRC.exists():
    sys.path.insert(0, str(SRC))


def run_cmd(
    cmd: list[str],
    *,
    env: dict[str, str] | None = None,
    cwd: Path | None = None,
    log_path: Path | None = None,
) -> None:
    print("+", shlex.join(cmd), flush=True)
    if log_path is None:
        subprocess.run(cmd, check=True, env=env, cwd=str(cwd) if cwd else None)
        return

    log_path.parent.mkdir(parents=True, exist_ok=True)
    print(f"[log] {log_path}", flush=True)
    proc = subprocess.Popen(
        cmd,
        env=env,
        cwd=str(cwd) if cwd else None,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    assert proc.stdout is not None
    with log_path.open("w") as f:
        for line in proc.stdout:
            print(line, end="")
            f.write(line)
    ret = proc.wait()
    if ret != 0:
        raise subprocess.CalledProcessError(ret, cmd)


REPO_ROOT: /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025


## 1) Configuration

Set:
- `DATA_MODE` stays `labeled_split` (this notebook forbids `data/test/`).
- `SELECTED_LABELED_SPLIT` should be `holdout_ets_sacrecoeur_stpeters`.
- `RUN_PIPELINE=True` to execute the pipeline from this notebook

The notebook writes everything under `outputs_notebook/<RUN_TAG>/`.


In [2]:

DATA_MODE = "labeled_split"  # 'labeled_split' only (do not use data/test)

RUN_PIPELINE = True          # set True to run the full pipeline
OVERWRITE = True             # set True to rebuild caches + overwrite outputs

RUN_TAG = "sota_holdout"
WORK_DIR = REPO_ROOT / "outputs_notebook" / RUN_TAG
WORK_DIR.mkdir(parents=True, exist_ok=True)

HF_ENDPOINT = os.environ.get("HF_ENDPOINT", "https://hf-mirror.com")

# COLMAP runner (docker recommended for CUDA dense MVS).
COLMAP_RUNNER = "docker"  # 'auto' | 'local' | 'docker'

# If True, also write demo HTML pages.
EXPORT_DEMOS = True

# ----- Labeled split configuration (only used when DATA_MODE='labeled_split') -----

LABELED_SPLITS = {
    "test_split": {
        "labels": REPO_ROOT / "data/test_split_labels.csv",
        "images": REPO_ROOT / "data/test_split",  # may be empty in this repo
    },
    "ets_stairs": {
        "labels": REPO_ROOT / "data/test_split_labels_ets_stairs.csv",
        "images": REPO_ROOT / "data/test_split_ets_stairs",
    },
    "holdout_ets_sacrecoeur_stpeters": {
        "labels": REPO_ROOT / "data/test_split_labels_holdout_ets_sacrecoeur_stpeters.csv",
        "images": REPO_ROOT / "data/test_split_holdout_ets_sacrecoeur_stpeters",
    },
}

SELECTED_LABELED_SPLIT = "holdout_ets_sacrecoeur_stpeters"

# Thresholds for pose-aware mAA (defined per dataset+scene).
GT_THRESHOLDS_CSV = REPO_ROOT / "data/train_thresholds.csv"

# ----- Derived paths -----

if DATA_MODE == "unlabeled_test":
    raise ValueError("Do not use data/test in this notebook. Use DATA_MODE='labeled_split'.")
elif DATA_MODE == "labeled_split":
    spec = LABELED_SPLITS[SELECTED_LABELED_SPLIT]
    GT_LABELS_CSV = Path(spec["labels"])
    images_root = Path(spec["images"])

    # Many repos store split *labels* but not split *image folders*.
    # This repo has the images under data/train/<dataset>/..., so we create a symlink farm under WORK_DIR.
    if images_root.exists() and any(p.is_dir() for p in images_root.iterdir()):
        DATA_TEST_ROOT = images_root
    else:
        import csv

        fallback = WORK_DIR / "eval_data"
        fallback.mkdir(parents=True, exist_ok=True)

        datasets = set()
        with Path(GT_LABELS_CSV).open(newline="") as f:
            r = csv.DictReader(f)
            for row in r:
                datasets.add(str(row["dataset"]))

        created = 0
        for ds in sorted(datasets):
            src = REPO_ROOT / "data/train" / ds
            dst = fallback / ds
            if dst.exists():
                continue
            if not src.exists():
                print(f"[warn] missing data/train/{ds}; skipping")
                continue
            try:
                dst.symlink_to(src, target_is_directory=True)
                created += 1
            except Exception as e:
                print(f"[warn] failed to symlink {dst} -> {src}: {e}")

        DATA_TEST_ROOT = fallback
        print(f"[ok] Using holdout image root: {fallback} (new links={created})")
else:
    raise ValueError(DATA_MODE)

# Safety: ensure we never point to data/test.
forbidden = (REPO_ROOT / "data/test").resolve()
resolved = DATA_TEST_ROOT.resolve()
if resolved == forbidden or forbidden in resolved.parents:
    raise ValueError(f"DATA_TEST_ROOT must not be inside {forbidden} (got: {DATA_TEST_ROOT})")

SUBMISSION_CSV = WORK_DIR / "submission.csv"
CLUSTERS_CSV = WORK_DIR / "cache" / "clusters_retrieval.csv"
CACHE_RETRIEVAL_ROOT = WORK_DIR / "cache_retrieval"
RECON_ROOT = WORK_DIR / "recon"
DEMO_DIR = WORK_DIR / "demo"

print("DATA_TEST_ROOT:", DATA_TEST_ROOT)
print("WORK_DIR      :", WORK_DIR)
print("SUBMISSION_CSV:", SUBMISSION_CSV)
print("CLUSTERS_CSV  :", CLUSTERS_CSV)
print("RECON_ROOT    :", RECON_ROOT)


DATA_TEST_ROOT: /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/data/test_split_holdout_ets_sacrecoeur_stpeters
WORK_DIR      : /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs_notebook/sota_holdout
SUBMISSION_CSV: /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs_notebook/sota_holdout/submission.csv
CLUSTERS_CSV  : /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs_notebook/sota_holdout/cache/clusters_retrieval.csv
RECON_ROOT    : /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs_notebook/sota_holdout/recon


## 2) Create a labeled-split image root

Some labeled split CSVs are provided without a corresponding `data/test_split.../` image folder.

If you want to run `DATA_MODE='labeled_split'` without copying images, this cell can create
`data/eval_test_split/<dataset> -> data/train/<dataset>` symlinks for all datasets referenced in the selected labels CSV.

Run this **once** if needed.


In [3]:
CREATE_EVAL_SYMLINKS = False  # set True if you need data/eval_test_split

if CREATE_EVAL_SYMLINKS:
    if GT_LABELS_CSV is None:
        raise ValueError("Set DATA_MODE='labeled_split' and choose SELECTED_LABELED_SPLIT first")

    import csv

    eval_root = REPO_ROOT / "data/eval_test_split"
    eval_root.mkdir(parents=True, exist_ok=True)

    datasets = set()
    with Path(GT_LABELS_CSV).open(newline='') as f:
        r = csv.DictReader(f)
        for row in r:
            datasets.add(str(row['dataset']))

    created = 0
    for ds in sorted(datasets):
        src = REPO_ROOT / "data/train" / ds
        dst = eval_root / ds
        if dst.exists():
            continue
        if not src.exists():
            print(f"[warn] missing data/train/{ds}; skipping")
            continue
        try:
            dst.symlink_to(src, target_is_directory=True)
            created += 1
        except Exception as e:
            print(f"[warn] failed to symlink {dst} -> {src}: {e}")

    print(f"[ok] eval root ready: {eval_root} (new links={created})")


## 3) Run the full pipeline (`--preset sota`)

This runs:
- retrieval training (if needed)
- embeddings + clustering
- SfM with matcher fallback (`colmap -> dino -> diffusion`)
- dense MVS + depth fallback
- demo export

Set `RUN_PIPELINE=True` in the config cell if you want to execute it.


In [4]:
env = os.environ.copy()
if HF_ENDPOINT:
    env["HF_ENDPOINT"] = str(HF_ENDPOINT)

# --- Train retrieval on TRAIN datasets only (exclude holdout datasets) ---
import csv

if GT_LABELS_CSV is None:
    raise ValueError("GT_LABELS_CSV must be set (use DATA_MODE='labeled_split')")

holdout_datasets: set[str] = set()
with Path(GT_LABELS_CSV).open(newline="") as f:
    r = csv.DictReader(f)
    for row in r:
        holdout_datasets.add(str(row["dataset"]))

train_root = REPO_ROOT / "data/train"
train_datasets = [p.name for p in sorted(train_root.iterdir()) if p.is_dir() and p.name not in holdout_datasets]

RETRIEVAL_OUTPUT_DIR = WORK_DIR / "retrieval_finetune_train_only"
RETRIEVAL_CKPT = RETRIEVAL_OUTPUT_DIR / "best.pt"

print("Holdout datasets:", sorted(holdout_datasets))
print("Train datasets  :", train_datasets)
print("Retrieval ckpt  :", RETRIEVAL_CKPT)

if RUN_PIPELINE:
    if OVERWRITE or not RETRIEVAL_CKPT.exists():
        train_log = WORK_DIR / "logs" / "train_retrieval.log"
        cmd_train = [
            sys.executable,
            "scripts/train_retrieval.py",
            "--data-root",
            str(train_root),
            "--labels-csv",
            str(REPO_ROOT / "data/train_labels.csv"),
            "--output-dir",
            str(RETRIEVAL_OUTPUT_DIR),
            "--model-id",
            "facebook/dinov2-small",
            "--embed-dim",
            "256",
            "--freeze-backbone",
        ]
        if HF_ENDPOINT:
            cmd_train += ["--hf-endpoint", str(HF_ENDPOINT)]
        for ds in train_datasets:
            cmd_train += ["--dataset", str(ds)]
        run_cmd(cmd_train, env=env, cwd=REPO_ROOT, log_path=train_log)
    else:
        print(f"[skip] retrieval training: checkpoint exists at {RETRIEVAL_CKPT}")

# --- Run full pipeline on HOLDOUT datasets (no data/test) ---
cmd = [
    sys.executable,
    str(REPO_ROOT / "scripts/run_pipeline.py"),
    "--preset",
    "sota",
    "--no-train",
    "--checkpoint",
    str(RETRIEVAL_CKPT),
    "--runner",
    str(COLMAP_RUNNER),
    "--data-test-root",
    str(DATA_TEST_ROOT),
    "--cache-root",
    str(CACHE_RETRIEVAL_ROOT),
    "--clusters-csv",
    str(CLUSTERS_CSV),
    "--recon-output-root",
    str(RECON_ROOT),
    "--submission-csv",
    str(SUBMISSION_CSV),
    "--demo-out-dir",
    str(DEMO_DIR),
]

if not EXPORT_DEMOS:
    cmd += ["--no-demo"]

if OVERWRITE:
    cmd += ["--overwrite"]

if RUN_PIPELINE:
    pipeline_log = WORK_DIR / "logs" / "run_pipeline.log"
    run_cmd(cmd, env=env, cwd=REPO_ROOT, log_path=pipeline_log)
else:
    print("Pipeline command (not executed):")
    print(shlex.join(cmd))


Holdout datasets: ['ETs', 'pt_sacrecoeur_trevi_tajmahal', 'pt_stpeters_stpauls']
Train datasets  : ['amy_gardens', 'fbk_vineyard', 'imc2023_haiper', 'imc2023_heritage', 'imc2023_theather_imc2024_church', 'imc2024_dioscuri_baalshamin', 'imc2024_lizard_pond', 'pt_brandenburg_british_buckingham', 'pt_piazzasanmarco_grandplace', 'stairs']
Retrieval ckpt  : /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs_notebook/sota_holdout/retrieval_finetune_train_only/best.pt
+ /home/shamika/miniconda3/envs/imc25/bin/python scripts/train_retrieval.py --data-root /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/data/train --labels-csv /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/data/train_labels.csv --output-dir /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs_notebook/sota_holdout/retrieval_finetune_train_only --model-id facebook/dinov2-small --embed-dim 256 --freeze-backbone --h

## 4) Load outputs

In [5]:
IMG_EXTS = {".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tif", ".tiff"}

# Ensure the output directories exist (the pipeline will create files inside them).
CLUSTERS_CSV.parent.mkdir(parents=True, exist_ok=True)
SUBMISSION_CSV.parent.mkdir(parents=True, exist_ok=True)

missing = []
if not CLUSTERS_CSV.exists():
    missing.append(str(CLUSTERS_CSV))
if not SUBMISSION_CSV.exists():
    missing.append(str(SUBMISSION_CSV))

if missing:
    print("[error] Required pipeline outputs not found:")
    for m in missing:
        print(" -", m)

    print()
    print("Next steps:")
    print("1) Go back to the config cell and set RUN_PIPELINE=True (and OVERWRITE=True if desired).")
    print("2) Run the pipeline cell (Section 3) and wait for it to finish.")
    print("3) Then re-run this cell and the rest of the notebook.")

    raise SystemExit("Missing pipeline outputs; run the pipeline first.")

clusters_df = pd.read_csv(CLUSTERS_CSV)
sub_df = pd.read_csv(SUBMISSION_CSV)

display(clusters_df.head())
display(sub_df.head())

print("clusters rows:", len(clusters_df))
print("submission rows:", len(sub_df))

# Basic sanity: submission should cover (at least) all clustered images.
if len(sub_df) != len(clusters_df):
    print("[warn] submission rows != clusters rows (this can happen if you post-process predictions)")


,dataset,image_id,scene
0,ETs,another_et_another_et001,cluster_0001
1,ETs,another_et_another_et002,cluster_0001
2,ETs,another_et_another_et003,cluster_0001
3,ETs,another_et_another_et004,cluster_0001
4,ETs,another_et_another_et005,cluster_0001


,image_id,dataset,scene,image,rotation_matrix,translation_vector
0,ETs_another_et_another_et001.png_public,ETs,cluster_0001,another_et_another_et001.png,0.996037218187;-0.0177709338712;-0.08714386895...,-1.98890453107;-1.37148964702;2.73618324819
1,ETs_another_et_another_et002.png_public,ETs,cluster_0001,another_et_another_et002.png,0.994558617222;-0.0174123261623;-0.10271303620...,-1.87210967629;-0.762569361992;1.44371056087
2,ETs_another_et_another_et003.png_public,ETs,cluster_0001,another_et_another_et003.png,0.997577696662;-0.0504247875199;-0.04791742821...,-2.02213371341;0.430391526327;-0.0390343245084
3,ETs_another_et_another_et004.png_public,ETs,cluster_0001,another_et_another_et004.png,0.997588666629;-0.0225421308098;-0.06564072327...,-1.94863625118;-0.916970800435;0.301323946343
4,ETs_another_et_another_et005.png_public,ETs,cluster_0001,another_et_another_et005.png,0.999979451715;-0.00237943304647;-0.0059526839...,-2.34170501192;-1.5001559813;1.50759926188


clusters rows: 447
submission rows: 447


## 5) Kaggle clustering metrics (mAA + Clustering Score + Final)

Based on the Kaggle competition, there are two main contributors to the (clustering) score: **Mean Average Accuracy (mAA)** and **Clustering Score (CS)**. The **Final Score** is the **Harmonic Mean** of these two values.

The ground truth data are compiled together in multiple datasets $D_k$. In each dataset there are one or more scenes $S_{ki}$, which contain images $I_{kiz}$. Additionally, each dataset may have a different number of scenes/images and may contain outlier images.

For each dataset, scenes are greedily assigned to clusters to maximize mAA (ties broken by higher CS). Predicted **outliers** clusters are excluded from this assignment.

This section also reports additional clustering metrics useful for a strong Results section: **F1**, **purity**, **fragmentation**, **ARI**, **NMI**, and **outlier precision/recall/F1**.

> Note: this section evaluates **scene clustering** only. For the pose-aware IMC25 metric (camera-center mAA), see the Pose-aware section later in the notebook.


In [6]:
# --- Clustering metrics (requires GT labels) ---
import math

OUTLIERS_LABEL = "outliers"
EXCLUDE_GT_OUTLIERS_FROM_SCORING = True  # recommended: treat GT outliers separately


def harmonic_mean(a: float, b: float) -> float:
    if a <= 0 or b <= 0:
        return 0.0
    return 2.0 / (1.0 / a + 1.0 / b)


def _comb2(x: np.ndarray) -> np.ndarray:
    return x * (x - 1) / 2.0


def adjusted_rand_index_from_contingency(n_ij: np.ndarray) -> float:
    # ARI computed from contingency table (rows=GT, cols=pred)
    n_ij = np.asarray(n_ij, dtype=np.float64)
    n = float(n_ij.sum())
    if n <= 1:
        return 0.0

    sum_comb = float(_comb2(n_ij).sum())
    a = n_ij.sum(axis=1)
    b = n_ij.sum(axis=0)
    sum_row = float(_comb2(a).sum())
    sum_col = float(_comb2(b).sum())
    total = float(_comb2(np.array([n], dtype=np.float64))[0])
    if total <= 0:
        return 0.0

    expected = (sum_row * sum_col) / total
    max_index = 0.5 * (sum_row + sum_col)
    denom = max_index - expected
    if denom <= 0:
        return 0.0
    return float((sum_comb - expected) / denom)


def normalized_mutual_information_from_contingency(n_ij: np.ndarray) -> float:
    # NMI = I(U;V) / sqrt(H(U) H(V))
    n_ij = np.asarray(n_ij, dtype=np.float64)
    n = float(n_ij.sum())
    if n <= 0:
        return 0.0

    a = n_ij.sum(axis=1)
    b = n_ij.sum(axis=0)

    # Entropies
    pa = a / n
    pb = b / n

    eps = 1e-12
    Ha = -float(np.sum(pa[pa > 0] * np.log(pa[pa > 0] + eps)))
    Hb = -float(np.sum(pb[pb > 0] * np.log(pb[pb > 0] + eps)))
    if Ha <= 0 or Hb <= 0:
        return 0.0

    # Mutual information
    I = 0.0
    for i in range(n_ij.shape[0]):
        for j in range(n_ij.shape[1]):
            nij = n_ij[i, j]
            if nij <= 0:
                continue
            I += (nij / n) * math.log((nij * n) / (a[i] * b[j] + eps) + eps)

    return float(I / math.sqrt(Ha * Hb))


def _ensure_str(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    out = df.copy()
    for c in cols:
        out[c] = out[c].astype(str)
    return out


if GT_LABELS_CSV is None:
    print("No GT labels (DATA_MODE='unlabeled_test'). Skipping clustering metrics.")
else:
    gt = pd.read_csv(Path(GT_LABELS_CSV))
    if not {"dataset", "scene", "image"}.issubset(set(gt.columns)):
        raise ValueError(f"GT_LABELS_CSV missing required columns: {GT_LABELS_CSV}")

    gt = gt[["dataset", "scene", "image"]].copy()
    gt["image_key"] = gt["image"].map(lambda x: Path(str(x)).stem)
    gt = gt.rename(columns={"scene": "gt_scene"})

    pred = clusters_df[["dataset", "image_id", "scene"]].copy()
    pred = pred.rename(columns={"image_id": "image_key", "scene": "pred_cluster"})

    gt = _ensure_str(gt, ["dataset", "gt_scene", "image_key"])
    pred = _ensure_str(pred, ["dataset", "pred_cluster", "image_key"])

    merged = gt.merge(pred, on=["dataset", "image_key"], how="left")
    merged["pred_cluster"] = merged["pred_cluster"].fillna(OUTLIERS_LABEL).astype(str)

    merged["gt_is_outlier"] = merged["gt_scene"].str.lower().eq(OUTLIERS_LABEL)
    merged["pred_is_outlier"] = merged["pred_cluster"].str.lower().eq(OUTLIERS_LABEL)

    # ---- Outlier detection metrics ----
    outlier_rows = []
    for dataset, g in merged.groupby("dataset", sort=True):
        gt_out = set(g.loc[g["gt_is_outlier"], "image_key"].tolist())
        pr_out = set(g.loc[g["pred_is_outlier"], "image_key"].tolist())
        tp = len(gt_out & pr_out)
        prec = tp / len(pr_out) if pr_out else 0.0
        rec = tp / len(gt_out) if gt_out else 0.0
        f1 = (2 * prec * rec / (prec + rec)) if (prec > 0 and rec > 0) else 0.0
        outlier_rows.append(
            {
                "dataset": str(dataset),
                "gt_outliers": int(len(gt_out)),
                "pred_outliers": int(len(pr_out)),
                "outlier_precision": float(prec),
                "outlier_recall": float(rec),
                "outlier_f1": float(f1),
            }
        )

    outlier_df = pd.DataFrame(outlier_rows).sort_values("dataset")
    display(outlier_df)

    # ---- Greedy scene->cluster assignment (clustering-only Kaggle metric) ----
    dataset_rows = []
    per_scene_rows = []
    per_cluster_rows = []
    frag_all_rows = []

    for dataset, g in merged.groupby("dataset", sort=True):
        g = g.copy()

        # Scenes to score
        g_scenes = g.loc[~g["gt_is_outlier"]].copy() if EXCLUDE_GT_OUTLIERS_FROM_SCORING else g.copy()

        gt_scenes = {sc: set(sg["image_key"].tolist()) for sc, sg in g_scenes.groupby("gt_scene", sort=True)}
        pred_clusters = {cl: set(cg["image_key"].tolist()) for cl, cg in g.groupby("pred_cluster", sort=True)}

        candidates = {k: v for k, v in pred_clusters.items() if k.lower() != OUTLIERS_LABEL and len(v) > 0}

        total_correct = 0
        total_scene_images = 0
        total_cluster_images = 0

        for scene, scene_imgs in sorted(gt_scenes.items()):
            n_scene = len(scene_imgs)
            total_scene_images += n_scene

            best_cluster = None
            best_maa = -1.0
            best_cs = -1.0
            best_correct = 0
            best_cluster_size = 0

            for c_label, c_imgs in candidates.items():
                inter = len(scene_imgs & c_imgs)
                maa = inter / n_scene if n_scene else 0.0
                cs = inter / len(c_imgs) if c_imgs else 0.0
                if (maa > best_maa) or (maa == best_maa and cs > best_cs) or (
                    maa == best_maa and cs == best_cs and (best_cluster is None or str(c_label) < str(best_cluster))
                ):
                    best_cluster = c_label
                    best_maa = maa
                    best_cs = cs
                    best_correct = inter
                    best_cluster_size = len(c_imgs)

            if best_cluster is None:
                best_cluster = "(none)"
                best_maa = 0.0
                best_cs = 0.0
                best_correct = 0
                best_cluster_size = 0

            total_correct += best_correct
            total_cluster_images += best_cluster_size

            f1 = (2 * best_maa * best_cs / (best_maa + best_cs)) if (best_maa > 0 and best_cs > 0) else 0.0

            per_scene_rows.append(
                {
                    "dataset": str(dataset),
                    "gt_scene": str(scene),
                    "assigned_cluster": str(best_cluster),
                    "n_scene": int(n_scene),
                    "n_cluster": int(best_cluster_size),
                    "n_correct": int(best_correct),
                    "mAA(scene)": float(best_maa),
                    "CS(scene)": float(best_cs),
                    "F1(scene)": float(f1),
                }
            )

        maa = (total_correct / total_scene_images) if total_scene_images else 0.0
        cs = (total_correct / total_cluster_images) if total_cluster_images else 0.0
        final = harmonic_mean(float(maa), float(cs))

        # Additional clustering summary metrics (label-based, no greedy assignment)
        # Build contingency over scored images.
        label_df = g_scenes[["gt_scene", "pred_cluster"]].copy()
        cont = pd.crosstab(label_df["gt_scene"], label_df["pred_cluster"])  # rows=GT scenes, cols=pred clusters
        nmi = normalized_mutual_information_from_contingency(cont.values)
        ari = adjusted_rand_index_from_contingency(cont.values)

        # Purity over predicted clusters (excluding predicted outliers)
        purity_rows = []
        for cl, cg in g.groupby("pred_cluster", sort=True):
            if str(cl).lower() == OUTLIERS_LABEL:
                continue
            size = int(len(cg))
            if size <= 0:
                continue
            vc = cg["gt_scene"].value_counts()
            top_scene = str(vc.index[0]) if len(vc) else "(none)"
            top_count = int(vc.iloc[0]) if len(vc) else 0
            purity = float(top_count / size) if size else 0.0
            purity_rows.append({"dataset": str(dataset), "pred_cluster": str(cl), "cluster_size": size, "top_gt_scene": top_scene, "top_count": top_count, "purity": purity})
        purity_df = pd.DataFrame(purity_rows)
        if len(purity_df) > 0:
            weighted_purity = float((purity_df["purity"] * purity_df["cluster_size"]).sum() / purity_df["cluster_size"].sum())
        else:
            weighted_purity = 0.0

        # Fragmentation: how many predicted clusters intersect each GT scene
        frag_rows = []
        for scene, sg in g_scenes.groupby("gt_scene", sort=True):
            # Count unique predicted clusters with >=1 image from the scene (excluding predicted outliers).
            clusters_hit = set(sg.loc[~sg["pred_is_outlier"], "pred_cluster"].tolist())
            frag_rows.append({"dataset": str(dataset), "gt_scene": str(scene), "n_pred_clusters_hit": int(len(clusters_hit)), "scene_size": int(len(sg))})
        frag_df = pd.DataFrame(frag_rows)
        frag_all_rows.extend(frag_rows)
        mean_fragmentation = float(frag_df["n_pred_clusters_hit"].mean()) if len(frag_df) else 0.0

        # Pred cluster counts
        n_pred_clusters = int(len({k for k in pred_clusters.keys() if str(k).lower() != OUTLIERS_LABEL}))

        # Join outlier metrics
        out_row = outlier_df.loc[outlier_df["dataset"] == str(dataset)].iloc[0].to_dict() if len(outlier_df) else {}

        dataset_rows.append(
            {
                "dataset": str(dataset),
                "mAA": float(maa),
                "clustering_score": float(cs),
                "final_score": float(final),
                "n_gt_scenes_scored": int(len(gt_scenes)),
                "n_images_in_scenes": int(total_scene_images),
                "n_pred_clusters": n_pred_clusters,
                "ARI": float(ari),
                "NMI": float(nmi),
                "weighted_purity": float(weighted_purity),
                "mean_fragmentation": float(mean_fragmentation),
                **{k: out_row.get(k) for k in ["gt_outliers", "pred_outliers", "outlier_precision", "outlier_recall", "outlier_f1"]},
            }
        )

        per_cluster_rows.append(purity_df)

    scores_df = pd.DataFrame(dataset_rows).sort_values("final_score", ascending=False)
    scenes_df = pd.DataFrame(per_scene_rows).sort_values(["dataset", "gt_scene"])
    frag_all_df = pd.DataFrame(frag_all_rows)
    purity_all = pd.concat([df for df in per_cluster_rows if isinstance(df, pd.DataFrame) and len(df) > 0], ignore_index=True) if per_cluster_rows else pd.DataFrame()

    print()
    print("Clustering scores (per dataset):")
    display(scores_df)

    mean_scores = {
        "mAA": float(scores_df["mAA"].mean()) if len(scores_df) else 0.0,
        "clustering_score": float(scores_df["clustering_score"].mean()) if len(scores_df) else 0.0,
        "final_score": float(scores_df["final_score"].mean()) if len(scores_df) else 0.0,
        "ARI": float(scores_df["ARI"].mean()) if len(scores_df) else 0.0,
        "NMI": float(scores_df["NMI"].mean()) if len(scores_df) else 0.0,
        "weighted_purity": float(scores_df["weighted_purity"].mean()) if len(scores_df) else 0.0,
        "mean_fragmentation": float(scores_df["mean_fragmentation"].mean()) if len(scores_df) else 0.0,
        "n_datasets": int(len(scores_df)),
    }
    mean_df = pd.DataFrame([mean_scores])
    print()
    print("Mean over datasets:")
    display(mean_df)

    # Export paper tables
    out_tables = WORK_DIR / "paper_tables_clustering"
    out_tables.mkdir(parents=True, exist_ok=True)
    scores_df.to_csv(out_tables / "per_dataset.csv", index=False)
    scenes_df.to_csv(out_tables / "per_scene.csv", index=False)
    mean_df.to_csv(out_tables / "mean.csv", index=False)
    if len(purity_all) > 0:
        purity_all.to_csv(out_tables / "per_cluster_purity.csv", index=False)

    # 1) Per-dataset bars
    # Extra plots for Results section
    print()
    print("Worst scenes by mAA (recall-like):")
    display(scenes_df.sort_values("mAA(scene)", ascending=True).head(15))
    print()
    print("Worst scenes by CS (precision-like):")
    display(scenes_df.sort_values("CS(scene)", ascending=True).head(15))

    fig = px.histogram(scenes_df, x="mAA(scene)", nbins=30, title="Distribution of per-scene mAA")
    fig.show()
    fig = px.histogram(scenes_df, x="CS(scene)", nbins=30, title="Distribution of per-scene CS")
    fig.show()
    fig = px.histogram(scenes_df, x="F1(scene)", nbins=30, title="Distribution of per-scene F1")
    fig.show()

    if len(frag_all_df) > 0:
        fig = px.histogram(frag_all_df, x="n_pred_clusters_hit", nbins=30, title="Scene fragmentation: #pred clusters hit per GT scene")
        fig.show()

    if len(outlier_df) > 0:
        fig = px.bar(
            outlier_df.sort_values("dataset"),
            x="dataset",
            y=["outlier_precision", "outlier_recall", "outlier_f1"],
            barmode="group",
            title="Outlier detection metrics (precision/recall/F1)",
        )
        fig.show()

    fig = px.bar(scores_df.sort_values("dataset"), x="dataset", y="final_score", title="Clustering Final Score per dataset")
    fig.show()
    fig = px.bar(scores_df.sort_values("dataset"), x="dataset", y=["mAA", "clustering_score"], barmode="group", title="Clustering mAA vs Clustering Score per dataset")
    fig.show()

    # 2) Scene-level scatter: mAA vs CS (bubble=size)
    fig = px.scatter(
        scenes_df,
        x="mAA(scene)",
        y="CS(scene)",
        color="dataset",
        size="n_scene",
        hover_data=["gt_scene", "assigned_cluster", "n_correct", "n_cluster"],
        title="Per-scene clustering quality (mAA vs CS)",
    )
    fig.show()

    # 3) Cluster purity distribution
    if len(purity_all) > 0:
        fig = px.histogram(purity_all, x="purity", nbins=30, title="Predicted cluster purity histogram")
        fig.show()
        fig = px.scatter(
            purity_all,
            x="cluster_size",
            y="purity",
            color="dataset",
            hover_data=["pred_cluster", "top_gt_scene", "top_count"],
            title="Cluster purity vs cluster size",
        )
        fig.update_xaxes(type="log")
        fig.show()

    # 4) Confusion heatmap + Sankey for one dataset
    DATASET_TO_VIZ = str(scores_df.sort_values("final_score", ascending=False).iloc[0]["dataset"]) if len(scores_df) else None
    if DATASET_TO_VIZ is not None:
        df_ds = merged[merged["dataset"] == DATASET_TO_VIZ].copy()
        if EXCLUDE_GT_OUTLIERS_FROM_SCORING:
            df_ds = df_ds[~df_ds["gt_is_outlier"]].copy()

        cm = pd.crosstab(df_ds["gt_scene"], df_ds["pred_cluster"])
        # Sort by size for readability
        cm = cm.loc[cm.sum(axis=1).sort_values(ascending=False).index]
        cm = cm[cm.sum(axis=0).sort_values(ascending=False).index]

        MAX_SCENES = 25
        MAX_CLUSTERS = 25
        cm_small = cm.iloc[:MAX_SCENES, :MAX_CLUSTERS]

        fig = px.imshow(
            cm_small,
            text_auto=True,
            color_continuous_scale="Blues",
            title=f"GT scene vs predicted cluster (counts) - {DATASET_TO_VIZ} (top {MAX_SCENES}x{MAX_CLUSTERS})",
        )
        fig.update_layout(height=650)
        fig.show()

        # Sankey (top links)
        links = (
            df_ds.groupby(["gt_scene", "pred_cluster"], as_index=False)
            .size()
            .rename(columns={"size": "count"})
            .sort_values("count", ascending=False)
        )
        top_links = links.head(80)
        scenes = top_links["gt_scene"].astype(str).unique().tolist()
        clusters = top_links["pred_cluster"].astype(str).unique().tolist()
        nodes = scenes + clusters
        node_index = {n: i for i, n in enumerate(nodes)}

        src = [node_index[s] for s in top_links["gt_scene"].astype(str).tolist()]
        tgt = [node_index[c] for c in top_links["pred_cluster"].astype(str).tolist()]
        val = top_links["count"].astype(int).tolist()

        fig = go.Figure(
            go.Sankey(
                node=dict(label=nodes, pad=12, thickness=14),
                link=dict(source=src, target=tgt, value=val),
            )
        )
        fig.update_layout(title=f"Scene->Cluster Sankey (top links) - {DATASET_TO_VIZ}", height=650)
        fig.show()

,dataset,gt_outliers,pred_outliers,outlier_precision,outlier_recall,outlier_f1
0,ETs,3,0,0.0,0.0,0.0
1,pt_sacrecoeur_trevi_tajmahal,0,4,0.0,0.0,0.0
2,pt_stpeters_stpauls,0,1,0.0,0.0,0.0



Clustering scores (per dataset):


,dataset,mAA,clustering_score,final_score,n_gt_scenes_scored,n_images_in_scenes,n_pred_clusters,ARI,NMI,weighted_purity,mean_fragmentation,gt_outliers,pred_outliers,outlier_precision,outlier_recall,outlier_f1
0,ETs,1.000000,0.863636,0.926829,2,19,2,1.000000,1.000000,0.863636,1.0,3,0,0.0,0.0,0.0
1,pt_sacrecoeur_trevi_tajmahal,0.715556,1.000000,0.834197,3,225,6,0.680929,0.801913,1.000000,2.0,0,4,0.0,0.0,0.0
2,pt_stpeters_stpauls,0.580000,1.000000,0.734177,2,200,5,0.479133,0.667491,0.994975,3.0,0,1,0.0,0.0,0.0



Mean over datasets:


,mAA,clustering_score,final_score,ARI,NMI,weighted_purity,mean_fragmentation,n_datasets
0,0.765185,0.954545,0.831734,0.72002,0.823135,0.95287,2.0,3



Worst scenes by mAA (recall-like):


,dataset,gt_scene,assigned_cluster,n_scene,n_cluster,n_correct,mAA(scene),CS(scene),F1(scene)
5,pt_stpeters_stpauls,st_pauls_cathedral,cluster_0002,100,55,55,0.550000,1.000000,0.709677
4,pt_sacrecoeur_trevi_tajmahal,trevi_fountain,cluster_0003,75,44,44,0.586667,1.000000,0.739496
6,pt_stpeters_stpauls,st_peters_square,cluster_0001,100,61,61,0.610000,1.000000,0.757764
2,pt_sacrecoeur_trevi_tajmahal,sacre_coeur,cluster_0002,75,46,46,0.613333,1.000000,0.760331
3,pt_sacrecoeur_trevi_tajmahal,taj_mahal,cluster_0001,75,71,71,0.946667,1.000000,0.972603
1,ETs,another_ET,cluster_0001,10,12,10,1.000000,0.833333,0.909091
0,ETs,ET,cluster_0002,9,10,9,1.000000,0.900000,0.947368



Worst scenes by CS (precision-like):


,dataset,gt_scene,assigned_cluster,n_scene,n_cluster,n_correct,mAA(scene),CS(scene),F1(scene)
1,ETs,another_ET,cluster_0001,10,12,10,1.000000,0.833333,0.909091
0,ETs,ET,cluster_0002,9,10,9,1.000000,0.900000,0.947368
2,pt_sacrecoeur_trevi_tajmahal,sacre_coeur,cluster_0002,75,46,46,0.613333,1.000000,0.760331
3,pt_sacrecoeur_trevi_tajmahal,taj_mahal,cluster_0001,75,71,71,0.946667,1.000000,0.972603
4,pt_sacrecoeur_trevi_tajmahal,trevi_fountain,cluster_0003,75,44,44,0.586667,1.000000,0.739496
5,pt_stpeters_stpauls,st_pauls_cathedral,cluster_0002,100,55,55,0.550000,1.000000,0.709677
6,pt_stpeters_stpauls,st_peters_square,cluster_0001,100,61,61,0.610000,1.000000,0.757764


## 6) Diagnostics metrics (works without GT)

In [7]:
def parse_floats_semicolon(s: str, expected: int) -> Optional[np.ndarray]:
    if not isinstance(s, str):
        return None
    s = s.strip()
    if not s or s.lower() == "nan":
        return None
    parts = s.split(";")
    if len(parts) != expected:
        return None
    arr = np.array([float(x) for x in parts], dtype=np.float64)
    if np.any(~np.isfinite(arr)):
        return None
    return arr


def parse_rotation(s: str) -> Optional[np.ndarray]:
    v = parse_floats_semicolon(s, 9)
    if v is None:
        return None
    return v.reshape(3, 3)


def parse_translation(s: str) -> Optional[np.ndarray]:
    return parse_floats_semicolon(s, 3)


def camera_center(R: np.ndarray, t: np.ndarray) -> np.ndarray:
    # COLMAP convention: x_cam = R * x_world + t
    return -R.T @ t


def _nan_pose_rows(sub: pd.DataFrame) -> pd.Series:
    # Pose is missing if rotation_matrix or translation_vector cannot be parsed.
    has = []
    for _, r in sub.iterrows():
        R = parse_rotation(str(r.get("rotation_matrix", "")))
        t = parse_translation(str(r.get("translation_vector", "")))
        has.append(R is None or t is None)
    return pd.Series(has)


sub_df = sub_df.copy()
sub_df["pose_missing"] = _nan_pose_rows(sub_df)
sub_df["has_pose"] = ~sub_df["pose_missing"]

# Pose coverage per dataset
cov = (
    sub_df.groupby("dataset", as_index=False)
    .agg(n_images=("image", "count"), n_pose=("has_pose", "sum"))
    .assign(pose_coverage=lambda d: d["n_pose"] / d["n_images"])
    .sort_values("pose_coverage", ascending=False)
)
display(cov)

fig = px.bar(cov, x="dataset", y="pose_coverage", title="Pose coverage (fraction with non-NaN pose)")
fig.show()

# Cluster stats
outliers_label = "outliers"
cl = clusters_df.copy()
cl["is_outlier"] = cl["scene"].astype(str).str.lower().eq(outliers_label)

cluster_sizes = (
    cl.loc[~cl["is_outlier"]]
    .groupby(["dataset", "scene"], as_index=False)
    .agg(n_images=("image_id", "count"))
    .sort_values(["dataset", "n_images"], ascending=[True, False])
)
display(cluster_sizes.head(20))

outlier_rates = (
    cl.groupby("dataset", as_index=False)
    .agg(n_images=("image_id", "count"), n_outliers=("is_outlier", "sum"))
    .assign(outlier_rate=lambda d: d["n_outliers"] / d["n_images"])
    .sort_values("outlier_rate", ascending=False)
)
display(outlier_rates)

fig = px.bar(outlier_rates, x="dataset", y="outlier_rate", title="Outlier rate per dataset")
fig.show()

# Cluster size distribution
fig = px.histogram(cluster_sizes, x="n_images", nbins=30, title="Cluster size histogram (excluding outliers)")
fig.show()


,dataset,n_images,n_pose,pose_coverage
0,ETs,22,19,0.863636
1,pt_sacrecoeur_trevi_tajmahal,225,0,0.000000
2,pt_stpeters_stpauls,200,0,0.000000


,dataset,scene,n_images
0,ETs,cluster_0001,12
1,ETs,cluster_0002,10
2,pt_sacrecoeur_trevi_tajmahal,cluster_0001,71
3,pt_sacrecoeur_trevi_tajmahal,cluster_0002,46
4,pt_sacrecoeur_trevi_tajmahal,cluster_0003,44
5,pt_sacrecoeur_trevi_tajmahal,cluster_0004,29
6,pt_sacrecoeur_trevi_tajmahal,cluster_0005,19
7,pt_sacrecoeur_trevi_tajmahal,cluster_0006,12
8,pt_stpeters_stpauls,cluster_0001,61
9,pt_stpeters_stpauls,cluster_0002,55


,dataset,n_images,n_outliers,outlier_rate
1,pt_sacrecoeur_trevi_tajmahal,225,4,0.017778
2,pt_stpeters_stpauls,200,1,0.005000
0,ETs,22,0,0.000000


## 7) Reconstruction stats (registered images + dense vertex counts)

In [8]:
def u64_head(path: Path) -> int:
    if not path.exists():
        return 0
    with path.open("rb") as f:
        head = f.read(8)
    if len(head) != 8:
        return 0
    (n,) = struct.unpack("<Q", head)
    return int(n)


def pick_best_sparse_model(sparse_root: Path) -> Path | None:
    if not sparse_root.exists():
        return None
    candidates = [p for p in sparse_root.iterdir() if p.is_dir() and p.name.isdigit()]
    if not candidates:
        return None

    best = None
    best_score = None
    for p in sorted(candidates, key=lambda x: int(x.name)):
        n_images = u64_head(p / "images.bin")
        n_points = u64_head(p / "points3D.bin")
        score = (n_images, n_points, int(p.name))
        if best_score is None or score > best_score:
            best_score = score
            best = p
    return best


def ply_vertex_count(ply_path: Path) -> int | None:
    if not ply_path.exists():
        return None
    try:
        with ply_path.open("rb") as f:
            # Read header as ASCII.
            header = b""
            while b"end_header" not in header and len(header) < 100_000:
                chunk = f.read(4096)
                if not chunk:
                    break
                header += chunk
            text = header.decode("ascii", errors="ignore")
        for line in text.splitlines():
            parts = line.strip().split()
            if len(parts) == 3 and parts[0] == "element" and parts[1] == "vertex":
                return int(parts[2])
    except Exception:
        return None
    return None


rows = []
for ds, scene in sorted({(r["dataset"], r["scene"]) for _, r in clusters_df.iterrows()}):
    if str(scene).lower() == "outliers":
        continue
    cluster_dir = RECON_ROOT / f"{ds}_{scene}"
    sparse_root = cluster_dir / "sparse"
    best = pick_best_sparse_model(sparse_root)

    n_reg = u64_head(best / "images.bin") if best else 0
    n_pts = u64_head(best / "points3D.bin") if best else 0

    ply = cluster_dir / "dense_points.ply"
    vtx = ply_vertex_count(ply)

    rows.append(
        {
            "dataset": ds,
            "scene": scene,
            "registered_images": int(n_reg),
            "sparse_points": int(n_pts),
            "dense_vertices": int(vtx) if vtx is not None else None,
            "cluster_dir": str(cluster_dir),
        }
    )

recon_df = pd.DataFrame(rows).sort_values(["dataset", "registered_images"], ascending=[True, False])
display(recon_df.head(30))

fig = px.histogram(recon_df, x="registered_images", nbins=30, title="Registered images per cluster (SfM)")
fig.show()

# If dense exists, show distribution
if recon_df["dense_vertices"].notna().any():
    fig = px.histogram(
        recon_df.dropna(subset=["dense_vertices"]),
        x="dense_vertices",
        nbins=40,
        title="Dense vertices per cluster (from PLY header)",
        log_y=True,
    )
    fig.show()


,dataset,scene,registered_images,sparse_points,dense_vertices,cluster_dir
0,ETs,cluster_0001,10,950,24765,/home/shamika/documents/prog_workspace/ml_imc2...
1,ETs,cluster_0002,9,1151,45149,/home/shamika/documents/prog_workspace/ml_imc2...
2,pt_sacrecoeur_trevi_tajmahal,cluster_0001,0,0,200000,/home/shamika/documents/prog_workspace/ml_imc2...
3,pt_sacrecoeur_trevi_tajmahal,cluster_0002,0,0,200000,/home/shamika/documents/prog_workspace/ml_imc2...
4,pt_sacrecoeur_trevi_tajmahal,cluster_0003,0,0,200000,/home/shamika/documents/prog_workspace/ml_imc2...
5,pt_sacrecoeur_trevi_tajmahal,cluster_0004,0,0,200000,/home/shamika/documents/prog_workspace/ml_imc2...
6,pt_sacrecoeur_trevi_tajmahal,cluster_0005,0,0,200000,/home/shamika/documents/prog_workspace/ml_imc2...
7,pt_sacrecoeur_trevi_tajmahal,cluster_0006,0,0,200000,/home/shamika/documents/prog_workspace/ml_imc2...
8,pt_stpeters_stpauls,cluster_0001,0,0,200000,/home/shamika/documents/prog_workspace/ml_imc2...
9,pt_stpeters_stpauls,cluster_0002,0,0,200000,/home/shamika/documents/prog_workspace/ml_imc2...


## 8) Embedding PCA visualization (per dataset)

In [9]:
import json

# Choose a dataset to visualize (must exist under CACHE_RETRIEVAL_ROOT).
DATASET_TO_PLOT = None  # e.g. 'ETs'; if None, pick the first cached dataset.

cache_datasets = sorted([p.name for p in CACHE_RETRIEVAL_ROOT.iterdir() if p.is_dir()]) if CACHE_RETRIEVAL_ROOT.exists() else []
if not cache_datasets:
    raise FileNotFoundError(f"No cache datasets found in {CACHE_RETRIEVAL_ROOT} (run pipeline first).")

if DATASET_TO_PLOT is None:
    DATASET_TO_PLOT = cache_datasets[0]

ds_dir = CACHE_RETRIEVAL_ROOT / DATASET_TO_PLOT
emb_path = ds_dir / "embeddings.npy"
meta_path = ds_dir / "meta.json"

emb = np.load(emb_path)
meta = json.loads(meta_path.read_text())
image_ids = meta.get("image_ids", [])

print("dataset:", DATASET_TO_PLOT, "embeddings:", emb.shape, "images:", len(image_ids))

# Map image_id -> predicted cluster
cl_map = (
    clusters_df.loc[clusters_df["dataset"] == DATASET_TO_PLOT, ["image_id", "scene"]]
    .astype(str)
    .set_index("image_id")["scene"]
    .to_dict()
)

labels = [cl_map.get(str(i), "(missing)") for i in image_ids]

# PCA via SVD (no sklearn dependency).
X = emb.astype(np.float64)
X = X - X.mean(axis=0, keepdims=True)
# Reduce to 2D
U, S, Vt = np.linalg.svd(X, full_matrices=False)
Z = U[:, :2] * S[:2]

pca_df = pd.DataFrame({"x": Z[:, 0], "y": Z[:, 1], "scene": labels, "image_id": image_ids})

fig = px.scatter(
    pca_df,
    x="x",
    y="y",
    color="scene",
    title=f"PCA of retrieval embeddings: {DATASET_TO_PLOT}",
    hover_data=["image_id"],
)
fig.show()


dataset: ETs embeddings: (22, 256) images: 22


## 9) Retrieval graph visualization (subsampled)

This uses the cached retrieval neighbor pairs to build a graph and show a spring-layout plot.
For large datasets, it subsamples nodes for responsiveness.


In [10]:
import networkx as nx

MAX_NODES_GRAPH = 250  # reduce if the plot is slow

# Find pairs file (topk can vary).
pair_files = sorted(ds_dir.glob("pairs_topk_K*.npz"))
if not pair_files:
    raise FileNotFoundError(f"No pairs_topk_K*.npz in {ds_dir}")
pairs_path = pair_files[0]

pairs = np.load(pairs_path)
a = pairs["a"].astype(int)
b = pairs["b"].astype(int)

n = len(image_ids)
idx = np.arange(n)
if n > MAX_NODES_GRAPH:
    # Deterministic subsample: keep first MAX_NODES_GRAPH
    idx = idx[:MAX_NODES_GRAPH]

idx_set = set(idx.tolist())

G = nx.Graph()
for i in idx:
    G.add_node(int(i))

for i, j in zip(a.tolist(), b.tolist(), strict=False):
    if i in idx_set and j in idx_set:
        G.add_edge(int(i), int(j))

print("graph nodes:", G.number_of_nodes(), "edges:", G.number_of_edges())

# Layout
pos = nx.spring_layout(G, seed=0, k=None)

xs = []
ys = []
colors = []
texts = []
for i in G.nodes():
    x, y = pos[i]
    xs.append(x)
    ys.append(y)
    scene = labels[int(i)]
    colors.append(scene)
    texts.append(f"{image_ids[int(i)]}<br>{scene}")

# Edge traces
ex = []
ey = []
for u, v in G.edges():
    x0, y0 = pos[u]
    x1, y1 = pos[v]
    ex += [x0, x1, None]
    ey += [y0, y1, None]

edge_trace = go.Scatter(
    x=ex,
    y=ey,
    mode="lines",
    line=dict(width=0.5, color="rgba(150,150,150,0.4)"),
    hoverinfo="skip",
    showlegend=False,
)

node_trace = go.Scatter(
    x=xs,
    y=ys,
    mode="markers",
    marker=dict(size=6),
    text=texts,
    hoverinfo="text",
    showlegend=False,
)

fig = go.Figure(data=[edge_trace, node_trace])
fig.update_layout(title=f"Retrieval graph (subsampled) - {DATASET_TO_PLOT}", height=700)
fig.show()


graph nodes: 22 edges: 231


## 10) 3D camera-center visualization (colored by predicted cluster)

This uses the `submission.csv` poses, converts them to camera centers, and plots a 3D scatter.


In [11]:
def compute_camera_centers(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    centers = []
    for _, r in out.iterrows():
        R = parse_rotation(str(r.get("rotation_matrix", "")))
        t = parse_translation(str(r.get("translation_vector", "")))
        if R is None or t is None:
            centers.append((np.nan, np.nan, np.nan))
        else:
            c = camera_center(R, t)
            centers.append((float(c[0]), float(c[1]), float(c[2])))
    cc = np.array(centers, dtype=np.float64)
    out["cx"], out["cy"], out["cz"] = cc[:, 0], cc[:, 1], cc[:, 2]
    return out


cams_df = compute_camera_centers(sub_df)

# Plot a single dataset for clarity.
DATASET_CAM = cams_df["dataset"].iloc[0]
cams_ds = cams_df[cams_df["dataset"] == DATASET_CAM].copy()
cams_ds = cams_ds[cams_ds["has_pose"]].copy()

print("dataset:", DATASET_CAM, "poses:", len(cams_ds))

fig = px.scatter_3d(
    cams_ds,
    x="cx",
    y="cy",
    z="cz",
    color="scene",
    hover_data=["image"],
    title=f"Camera centers (pred) - {DATASET_CAM}",
)
fig.update_traces(marker=dict(size=3))
fig.show()


dataset: ETs poses: 19


## 11) (Optional) Point cloud preview for a cluster

Requires `open3d` (`pip install -e .[pointcloud]`).


In [12]:
CLUSTER_TO_PREVIEW = None  # e.g. 'ETs_cluster_0001'
MAX_POINTS = 120_000

# Pick a cluster with dense_points.ply
if CLUSTER_TO_PREVIEW is None:
    candidates = sorted([p for p in RECON_ROOT.iterdir() if p.is_dir() and (p / 'dense_points.ply').exists()]) if RECON_ROOT.exists() else []
    if candidates:
        CLUSTER_TO_PREVIEW = candidates[0].name

if CLUSTER_TO_PREVIEW is None:
    print("No dense point clouds found under", RECON_ROOT)
else:
    ply_path = RECON_ROOT / CLUSTER_TO_PREVIEW / "dense_points.ply"
    print("preview:", ply_path)

    try:
        import open3d as o3d  # type: ignore

        pcd = o3d.io.read_point_cloud(str(ply_path))
        pts = np.asarray(pcd.points, dtype=np.float64)
        cols = np.asarray(pcd.colors, dtype=np.float64) if pcd.has_colors() else None

        if pts.shape[0] > MAX_POINTS:
            idx = np.random.default_rng(0).choice(np.arange(pts.shape[0]), size=MAX_POINTS, replace=False)
            pts = pts[idx]
            if cols is not None:
                cols = cols[idx]

        pc_df = pd.DataFrame({"x": pts[:, 0], "y": pts[:, 1], "z": pts[:, 2]})
        if cols is not None and cols.shape[0] == pts.shape[0]:
            pc_df["r"] = cols[:, 0]
            pc_df["g"] = cols[:, 1]
            pc_df["b"] = cols[:, 2]
            pc_df["rgb"] = (
                (pc_df[["r", "g", "b"]].clip(0, 1) * 255)
                .round()
                .astype(int)
                .apply(lambda r: f"rgb({r[0]},{r[1]},{r[2]})", axis=1)
            )
            fig = go.Figure(
                data=[
                    go.Scatter3d(
                        x=pc_df["x"],
                        y=pc_df["y"],
                        z=pc_df["z"],
                        mode="markers",
                        marker=dict(size=1, color=pc_df["rgb"], opacity=0.8),
                    )
                ]
            )
        else:
            fig = px.scatter_3d(pc_df, x="x", y="y", z="z", title=f"Point cloud preview: {CLUSTER_TO_PREVIEW}")
            fig.update_traces(marker=dict(size=1, opacity=0.7))

        fig.update_layout(height=700)
        fig.show()
    except ImportError:
        print("open3d not installed. Install with: pip install -e .[pointcloud]")
    except Exception as e:
        print("failed to load point cloud:", e)


preview: /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs_notebook/sota_holdout/recon/ETs_cluster_0001/dense_points.ply
Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


/tmp/ipykernel_52952/790574013.py:38: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`



## 12) Pose-aware IMC25 metrics (only if GT labels are available)

If `DATA_MODE='labeled_split'`, this section computes:
- pose-aware mAA (camera centers, similarity aligned)
- clustering score
- final score = harmonic mean

Implementation matches `notebooks/08_pose_scoring_metrics.ipynb`.


In [13]:
def harmonic_mean(a: float, b: float) -> float:
    if a <= 0 or b <= 0:
        return 0.0
    return 2.0 / (1.0 / a + 1.0 / b)


def umeyama_similarity(X: np.ndarray, Y: np.ndarray, *, with_scale: bool = True) -> Tuple[float, np.ndarray, np.ndarray]:
    if X.shape != Y.shape or X.ndim != 2 or X.shape[1] != 3:
        raise ValueError("X and Y must have shape (N,3) and match")
    n = X.shape[0]
    if n < 3:
        raise ValueError("Need at least 3 points for a stable 3D similarity")

    muX = X.mean(axis=0)
    muY = Y.mean(axis=0)
    Xc = X - muX
    Yc = Y - muY

    cov = (Yc.T @ Xc) / n
    U, D, Vt = np.linalg.svd(cov)
    S = np.eye(3)
    if np.linalg.det(U) * np.linalg.det(Vt) < 0:
        S[-1, -1] = -1
    R = U @ S @ Vt

    if with_scale:
        varX = (Xc**2).sum(axis=1).mean()
        scale = float(np.trace(np.diag(D) @ S) / varX) if varX > 0 else 1.0
    else:
        scale = 1.0

    t = muY - scale * (R @ muX)
    return scale, R, t


def estimate_similarity_ransac(
    X: np.ndarray,
    Y: np.ndarray,
    *,
    max_iters: int = 512,
    inlier_threshold: float,
    seed: int = 0,
) -> Optional[Tuple[float, np.ndarray, np.ndarray, np.ndarray]]:
    if X.shape[0] < 3:
        return None

    rng = np.random.default_rng(seed)
    n = X.shape[0]

    best = None
    best_inliers = -1
    best_median = np.inf

    idx = np.arange(n)
    for _ in range(max_iters):
        sample = rng.choice(idx, size=3, replace=False)
        try:
            s, R, t = umeyama_similarity(X[sample], Y[sample], with_scale=True)
        except Exception:
            continue
        X_aligned = (s * (R @ X.T)).T + t
        err = np.linalg.norm(Y - X_aligned, axis=1)
        inliers = err < inlier_threshold
        cnt = int(inliers.sum())
        med = float(np.median(err[inliers])) if cnt > 0 else np.inf
        if (cnt > best_inliers) or (cnt == best_inliers and med < best_median):
            best_inliers = cnt
            best_median = med
            best = (s, R, t, inliers)

    if best is None or best_inliers < 3:
        return None

    s0, R0, t0, inliers = best
    try:
        s, R, t = umeyama_similarity(X[inliers], Y[inliers], with_scale=True)
    except Exception:
        s, R, t = s0, R0, t0

    return s, R, t, inliers


def normalize_image_key(dataset: str, *, image_id: Optional[str] = None, image: Optional[str] = None) -> str:
    if isinstance(image, str) and image.strip() and image.lower() != "nan":
        return Path(image).stem
    if not isinstance(image_id, str) or not image_id.strip():
        raise ValueError("Need either `image` or `image_id` to derive an image key")

    s = image_id.strip()
    if s.endswith("_public") or s.endswith("_private"):
        s = s.rsplit("_", 1)[0]
    prefix = f"{dataset}_"
    if s.startswith(prefix):
        s = s[len(prefix) :]
    return Path(s).stem


def load_ground_truth(labels_csv: Path, thresholds_csv: Path) -> Tuple[
    Dict[str, Dict[str, set[str]]],
    Dict[Tuple[str, str], np.ndarray],
    Dict[Tuple[str, str], np.ndarray],
    Dict[str, set[str]],
]:
    gt = pd.read_csv(labels_csv)
    required = {"dataset", "scene", "image", "rotation_matrix", "translation_vector"}
    missing = required - set(gt.columns)
    if missing:
        raise ValueError(f"Ground truth missing columns: {sorted(missing)}")

    gt = gt[list(required)].copy()
    gt["image_key"] = gt["image"].map(lambda x: Path(str(x)).stem)

    scenes: Dict[str, Dict[str, set[str]]] = {}
    centers: Dict[Tuple[str, str], np.ndarray] = {}
    for _, r in gt.iterrows():
        ds = str(r["dataset"])
        sc = str(r["scene"])
        k = str(r["image_key"])
        scenes.setdefault(ds, {}).setdefault(sc, set()).add(k)

        R = parse_rotation(str(r["rotation_matrix"]))
        t = parse_translation(str(r["translation_vector"]))
        if R is None or t is None:
            continue
        centers[(ds, k)] = camera_center(R, t)

    thr = pd.read_csv(thresholds_csv)
    if not {"dataset", "scene", "thresholds"}.issubset(thr.columns):
        raise ValueError("thresholds CSV must have columns: dataset, scene, thresholds")
    thresholds: Dict[Tuple[str, str], np.ndarray] = {}
    for _, r in thr.iterrows():
        ds = str(r["dataset"])
        sc = str(r["scene"])
        parts = [float(x) for x in str(r["thresholds"]).split(";")]
        parts = [x for x in parts if x > 0]
        thresholds[(ds, sc)] = np.array(parts, dtype=np.float64)

    all_images: Dict[str, set[str]] = {}
    for dataset, g in gt.groupby("dataset", sort=True):
        all_images[str(dataset)] = set(g["image_key"].astype(str).tolist())

    return scenes, centers, thresholds, all_images


def load_predictions_with_poses(
    pred_csv: Path,
    expected_images: Dict[str, set[str]],
    *,
    outliers_label: str = "outliers",
) -> Tuple[
    Dict[str, Dict[str, set[str]]],
    Dict[Tuple[str, str], Optional[np.ndarray]],
]:
    df = pd.read_csv(pred_csv)
    cols = set(df.columns)
    if "dataset" not in cols:
        raise ValueError("Predictions CSV must contain a `dataset` column")

    cluster_col = None
    for c in ["scene", "cluster", "cluster_label"]:
        if c in cols:
            cluster_col = c
            break
    if cluster_col is None:
        raise ValueError("Predictions CSV must contain one of: scene / cluster / cluster_label")

    image_col = "image" if "image" in cols else None
    image_id_col = "image_id" if "image_id" in cols else None
    if image_col is None and image_id_col is None:
        raise ValueError("Predictions CSV must contain `image` or `image_id`")

    has_pose = ("rotation_matrix" in cols) and ("translation_vector" in cols)

    keep_cols = ["dataset", cluster_col]
    if image_col:
        keep_cols.append(image_col)
    if image_id_col:
        keep_cols.append(image_id_col)
    if has_pose:
        keep_cols += ["rotation_matrix", "translation_vector"]

    df = df[keep_cols].copy().rename(columns={cluster_col: "pred_cluster"})

    def _row_to_key(r) -> str:
        return normalize_image_key(
            str(r["dataset"]),
            image_id=str(r[image_id_col]) if image_id_col and pd.notna(r[image_id_col]) else None,
            image=str(r[image_col]) if image_col and pd.notna(r[image_col]) else None,
        )

    df["image_key"] = df.apply(_row_to_key, axis=1)

    # Keep only images we expect.
    kept = []
    for dataset, g in df.groupby("dataset", sort=False):
        exp = expected_images.get(str(dataset), set())
        kept.append(g[g["image_key"].isin(exp)] if exp else g)
    df = pd.concat(kept, ignore_index=True) if kept else df

    # If predictions do not cover all images, treat missing ones as predicted outliers.
    rows_missing = []
    for dataset, exp in expected_images.items():
        present = set(df.loc[df["dataset"] == dataset, "image_key"].tolist())
        missing = sorted(exp - present)
        rows_missing.extend({"dataset": dataset, "pred_cluster": outliers_label, "image_key": k} for k in missing)
    if rows_missing:
        df = pd.concat([df, pd.DataFrame(rows_missing)], ignore_index=True)

    clusters: Dict[str, Dict[str, set[str]]] = {}
    for (dataset, cluster), g in df.groupby(["dataset", "pred_cluster"], sort=True):
        clusters.setdefault(str(dataset), {})[str(cluster)] = set(g["image_key"].tolist())

    pred_centers: Dict[Tuple[str, str], Optional[np.ndarray]] = {}
    if has_pose:
        for _, r in df.iterrows():
            ds = str(r["dataset"])
            k = str(r["image_key"])
            R = parse_rotation(str(r.get("rotation_matrix", "")))
            t = parse_translation(str(r.get("translation_vector", "")))
            if R is None or t is None:
                pred_centers[(ds, k)] = None
            else:
                pred_centers[(ds, k)] = camera_center(R, t)
    else:
        for _, r in df.iterrows():
            ds = str(r["dataset"])
            k = str(r["image_key"])
            pred_centers[(ds, k)] = None

    return clusters, pred_centers


def pose_mAA_for_scene_and_cluster(
    *,
    dataset: str,
    scene: str,
    scene_images: set[str],
    cluster_images: set[str],
    gt_centers: Dict[Tuple[str, str], np.ndarray],
    pred_centers: Dict[Tuple[str, str], Optional[np.ndarray]],
    thresholds: np.ndarray,
    min_align_points: int = 3,
    use_ransac: bool = True,
    ransac_iters: int = 512,
) -> Tuple[float, int, int]:
    inter = scene_images & cluster_images
    if not inter:
        return 0.0, 0, len(scene_images)

    X = []
    Y = []
    for k in inter:
        pc = pred_centers.get((dataset, k))
        gc = gt_centers.get((dataset, k))
        if pc is None or gc is None:
            continue
        X.append(pc)
        Y.append(gc)

    if len(X) < min_align_points:
        return 0.0, len(X), len(scene_images)

    X = np.stack(X, axis=0)
    Y = np.stack(Y, axis=0)

    max_thr = float(np.max(thresholds)) if thresholds.size else 0.0
    if max_thr <= 0:
        return 0.0, len(X), len(scene_images)

    if use_ransac:
        est = estimate_similarity_ransac(X, Y, max_iters=ransac_iters, inlier_threshold=max_thr, seed=0)
        if est is None:
            return 0.0, len(X), len(scene_images)
        s, R, t, _inliers = est
    else:
        s, R, t = umeyama_similarity(X, Y, with_scale=True)

    errors = np.full((len(scene_images),), np.inf, dtype=np.float64)
    scene_list = sorted(scene_images)
    for i, k in enumerate(scene_list):
        if k not in cluster_images:
            continue
        pc = pred_centers.get((dataset, k))
        gc = gt_centers.get((dataset, k))
        if pc is None or gc is None:
            continue
        pa = (s * (R @ pc)) + t
        errors[i] = float(np.linalg.norm(gc - pa))

    accs = [(errors < thr).mean() for thr in thresholds]
    return float(np.mean(accs)), len(X), len(scene_images)


@dataclass(frozen=True)
class DatasetScore:
    dataset: str
    pose_mAA: float
    clustering_score: float
    final_score: float
    n_scenes: int
    n_images_in_scenes: int


def score_dataset_pose_aware(
    *,
    dataset: str,
    gt_scenes: Dict[str, set[str]],
    pred_clusters: Dict[str, set[str]],
    gt_centers: Dict[Tuple[str, str], np.ndarray],
    pred_centers: Dict[Tuple[str, str], Optional[np.ndarray]],
    thresholds_by_scene: Dict[Tuple[str, str], np.ndarray],
    outliers_label: str = "outliers",
    min_align_points: int = 3,
    use_ransac: bool = True,
) -> Tuple[DatasetScore, pd.DataFrame]:
    candidates = {k: v for k, v in pred_clusters.items() if k != outliers_label and len(v) > 0}

    rows = []
    total_scene_images = 0
    total_correct = 0
    total_cluster_images = 0
    pose_mAA_weighted_sum = 0.0

    for scene, scene_imgs in sorted(gt_scenes.items()):
        n_scene = len(scene_imgs)
        total_scene_images += n_scene
        thresholds = thresholds_by_scene.get((dataset, scene), np.array([], dtype=np.float64))

        best_cluster = None
        best_pose_maa = -1.0
        best_cs = -1.0
        best_correct = 0
        best_cluster_size = 0
        best_align_pts = 0

        for c_label, c_imgs in candidates.items():
            inter = len(scene_imgs & c_imgs)
            if inter == 0:
                continue
            cs = inter / len(c_imgs) if c_imgs else 0.0
            pose_maa, n_align, _ = pose_mAA_for_scene_and_cluster(
                dataset=dataset,
                scene=scene,
                scene_images=scene_imgs,
                cluster_images=c_imgs,
                gt_centers=gt_centers,
                pred_centers=pred_centers,
                thresholds=thresholds,
                min_align_points=min_align_points,
                use_ransac=use_ransac,
            )
            if (pose_maa > best_pose_maa) or (pose_maa == best_pose_maa and cs > best_cs):
                best_cluster = c_label
                best_pose_maa = pose_maa
                best_cs = cs
                best_correct = inter
                best_cluster_size = len(c_imgs)
                best_align_pts = n_align

        if best_cluster is None:
            best_cluster = "(none)"
            best_pose_maa = 0.0
            best_cs = 0.0
            best_correct = 0
            best_cluster_size = 0
            best_align_pts = 0

        pose_mAA_weighted_sum += best_pose_maa * n_scene
        total_correct += best_correct
        total_cluster_images += best_cluster_size

        rows.append(
            {
                "dataset": dataset,
                "gt_scene": scene,
                "assigned_cluster": best_cluster,
                "pose_mAA(scene)": best_pose_maa,
                "CS(scene)": best_cs,
                "n_scene": n_scene,
                "n_cluster": best_cluster_size,
                "n_correct": best_correct,
                "n_align_pts": best_align_pts,
            }
        )

    pose_mAA = (pose_mAA_weighted_sum / total_scene_images) if total_scene_images else 0.0
    cs = (total_correct / total_cluster_images) if total_cluster_images else 0.0
    final = harmonic_mean(pose_mAA, cs)

    return (
        DatasetScore(
            dataset=dataset,
            pose_mAA=float(pose_mAA),
            clustering_score=float(cs),
            final_score=float(final),
            n_scenes=len(gt_scenes),
            n_images_in_scenes=int(total_scene_images),
        ),
        pd.DataFrame(rows),
    )


if GT_LABELS_CSV is None:
    print("GT_LABELS_CSV not set (DATA_MODE is unlabeled). Skipping official metrics.")
else:
    gt_scenes_by_dataset, gt_centers, thresholds_by_scene, all_images_by_dataset = load_ground_truth(
        labels_csv=Path(GT_LABELS_CSV),
        thresholds_csv=GT_THRESHOLDS_CSV,
    )
    pred_clusters_by_dataset, pred_centers = load_predictions_with_poses(
        pred_csv=SUBMISSION_CSV,
        expected_images=all_images_by_dataset,
        outliers_label="outliers",
    )

    datasets = sorted(gt_scenes_by_dataset.keys())

    ds_rows = []
    scene_rows = []
    for ds in datasets:
        ds_score, per_scene = score_dataset_pose_aware(
            dataset=ds,
            gt_scenes=gt_scenes_by_dataset[ds],
            pred_clusters=pred_clusters_by_dataset.get(ds, {}),
            gt_centers=gt_centers,
            pred_centers=pred_centers,
            thresholds_by_scene=thresholds_by_scene,
            outliers_label="outliers",
            min_align_points=3,
            use_ransac=True,
        )
        ds_rows.append(ds_score.__dict__)
        scene_rows.append(per_scene)

    ds_df = pd.DataFrame(ds_rows).sort_values("final_score", ascending=False)
    scene_df = pd.concat(scene_rows, ignore_index=True) if scene_rows else pd.DataFrame()

    mean_row = {
        "pose_mAA": float(ds_df["pose_mAA"].mean()) if len(ds_df) else 0.0,
        "clustering_score": float(ds_df["clustering_score"].mean()) if len(ds_df) else 0.0,
        "final_score": float(ds_df["final_score"].mean()) if len(ds_df) else 0.0,
        "n_datasets": int(len(ds_df)),
    }
    mean_df = pd.DataFrame([mean_row])

    display(ds_df)
    display(mean_df)

    fig = px.bar(ds_df, x="dataset", y="final_score", title="Final score per dataset (labeled split)")
    fig.show()
    fig = px.bar(
        ds_df.sort_values("dataset"),
        x="dataset",
        y=["pose_mAA", "clustering_score"],
        barmode="group",
        title="Pose-mAA vs Clustering Score per dataset",
    )
    fig.show()

    fig = px.scatter(
        ds_df,
        x="pose_mAA",
        y="clustering_score",
        text="dataset",
        title="Dataset trade-off: pose-mAA vs clustering score",
    )
    fig.update_traces(textposition="top center")
    fig.show()

    if len(scene_df) > 0:
        fig = px.scatter(
            scene_df,
            x="pose_mAA(scene)",
            y="CS(scene)",
            color="dataset",
            size="n_scene",
            hover_data=["gt_scene", "assigned_cluster", "n_correct", "n_cluster", "n_align_pts"],
            title="Per-scene pose-mAA vs CS (bubble=size of GT scene)",
        )
        fig.show()

        fig = px.histogram(scene_df, x="pose_mAA(scene)", nbins=30, title="Distribution of per-scene pose-mAA")
        fig.show()
        fig = px.histogram(scene_df, x="CS(scene)", nbins=30, title="Distribution of per-scene CS")
        fig.show()
        fig = px.histogram(scene_df, x="n_align_pts", nbins=30, title="Alignment points used (n_align_pts) histogram")
        fig.show()

        print()
        print("Worst scenes by pose-mAA:")
        display(scene_df.sort_values("pose_mAA(scene)", ascending=True).head(15))
        print()
        print("Worst scenes by CS:")
        display(scene_df.sort_values("CS(scene)", ascending=True).head(15))


    # Export tables
    out_tables = WORK_DIR / "paper_tables"
    out_tables.mkdir(parents=True, exist_ok=True)
    ds_df.to_csv(out_tables / "per_dataset.csv", index=False)
    scene_df.to_csv(out_tables / "per_scene.csv", index=False)
    mean_df.to_csv(out_tables / "mean.csv", index=False)

    latex_df = ds_df[["dataset", "final_score", "pose_mAA", "clustering_score", "n_scenes", "n_images_in_scenes"]].copy()
    (out_tables / "per_dataset.tex").write_text(latex_df.to_latex(index=False, float_format=lambda x: f"{x:.4f}"))
    (out_tables / "mean.tex").write_text(mean_df.to_latex(index=False, float_format=lambda x: f"{x:.4f}"))

    print("[ok] wrote metrics tables to", out_tables)


,dataset,pose_mAA,clustering_score,final_score,n_scenes,n_images_in_scenes
0,ETs,0.227273,0.617647,0.332278,3,22
1,pt_sacrecoeur_trevi_tajmahal,0.000000,1.000000,0.000000,3,225
2,pt_stpeters_stpauls,0.000000,1.000000,0.000000,2,200


,pose_mAA,clustering_score,final_score,n_datasets
0,0.075758,0.872549,0.110759,3



Worst scenes by pose-mAA:


,dataset,gt_scene,assigned_cluster,pose_mAA(scene),CS(scene),n_scene,n_cluster,n_correct,n_align_pts
3,pt_sacrecoeur_trevi_tajmahal,sacre_coeur,cluster_0002,0.000000,1.000000,75,46,46,0
2,ETs,outliers,cluster_0001,0.000000,0.166667,3,12,2,0
5,pt_sacrecoeur_trevi_tajmahal,trevi_fountain,cluster_0003,0.000000,1.000000,75,44,44,0
4,pt_sacrecoeur_trevi_tajmahal,taj_mahal,cluster_0001,0.000000,1.000000,75,71,71,0
6,pt_stpeters_stpauls,st_pauls_cathedral,cluster_0002,0.000000,1.000000,100,55,55,0
7,pt_stpeters_stpauls,st_peters_square,cluster_0001,0.000000,1.000000,100,61,61,0
1,ETs,another_ET,cluster_0001,0.183333,0.833333,10,12,10,10
0,ETs,ET,cluster_0002,0.351852,0.900000,9,10,9,9



Worst scenes by CS:


,dataset,gt_scene,assigned_cluster,pose_mAA(scene),CS(scene),n_scene,n_cluster,n_correct,n_align_pts
2,ETs,outliers,cluster_0001,0.000000,0.166667,3,12,2,0
1,ETs,another_ET,cluster_0001,0.183333,0.833333,10,12,10,10
0,ETs,ET,cluster_0002,0.351852,0.900000,9,10,9,9
3,pt_sacrecoeur_trevi_tajmahal,sacre_coeur,cluster_0002,0.000000,1.000000,75,46,46,0
4,pt_sacrecoeur_trevi_tajmahal,taj_mahal,cluster_0001,0.000000,1.000000,75,71,71,0
5,pt_sacrecoeur_trevi_tajmahal,trevi_fountain,cluster_0003,0.000000,1.000000,75,44,44,0
6,pt_stpeters_stpauls,st_pauls_cathedral,cluster_0002,0.000000,1.000000,100,55,55,0
7,pt_stpeters_stpauls,st_peters_square,cluster_0001,0.000000,1.000000,100,61,61,0


[ok] wrote metrics tables to /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs_notebook/sota_holdout/paper_tables


## 13) Links: interactive demos and explorers

- Demo HTML: `WORK_DIR/demo/index.html`
- Dash apps:
  - Jigsaw Explorer: `python scripts/jigsaw_explorer.py`
  - Pose Explorer: `python scripts/pose_explorer.py --csv <submission.csv> --images-root <...> --pointcloud <...>`


In [14]:
print("Demo dir:", DEMO_DIR)
index = DEMO_DIR / "index.html"
if index.exists():
    print("Demo index:", index)
else:
    print("No demo index found (did you run with EXPORT_DEMOS=True and preset sota?)")

print("\nRun Jigsaw Explorer:")
print("  python scripts/jigsaw_explorer.py")

print("\nRun Pose Explorer (example):")
example_cluster = None
if RECON_ROOT.exists():
    for p in sorted(RECON_ROOT.iterdir()):
        if p.is_dir() and (p / "dense_points.ply").exists():
            example_cluster = p
            break

if example_cluster is not None:
    print(
        "  python scripts/pose_explorer.py --csv "
        + str(SUBMISSION_CSV)
        + " --images-root "
        + str(example_cluster / "images")
        + " --pointcloud "
        + str(example_cluster / "dense_points.ply")
    )
else:
    print("  python scripts/pose_explorer.py --csv submission.csv --images-root <cluster/images> --pointcloud <cluster/dense_points.ply>")


Demo dir: /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs_notebook/sota_holdout/demo
Demo index: /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs_notebook/sota_holdout/demo/index.html

Run Jigsaw Explorer:
  python scripts/jigsaw_explorer.py

Run Pose Explorer (example):
  python scripts/pose_explorer.py --csv /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs_notebook/sota_holdout/submission.csv --images-root /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs_notebook/sota_holdout/recon/ETs_cluster_0001/images --pointcloud /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs_notebook/sota_holdout/recon/ETs_cluster_0001/dense_points.ply
